# Spoofing an architecture in Clustered Federated Learning

**The setting.** Clients differ in *what model they can run*. A hospital with a GPU
trains a convolutional network; a clinic on a laptop runs a small dense one. This is
**model heterogeneity**, and a federated server cannot average their weights together --
the tensors do not line up. So it groups clients by the architecture they **declare** and
keeps one global model per architecture.

**The attack.** That declaration is an unverified string. I run `mlp_flat`. I say I run
`dnn_flat`. The server seats me in the `dnn_flat` cluster, sends me its model, and averages
my updates into it.

**The question this notebook answers.** Does the lie get me in, what stops it if anything,
and what does it cost the cluster I invaded?

> This is an attack analysis. No defence is proposed. Where visibility is measured, that
> describes the attack's signature, not a countermeasure.

**Companion notebook:** `cfl-distribution.ipynb` attacks the declared *label histogram*.
Part 9 here compares the two, and the comparison is the point of running both.

## Why this is a separate axis, not "Part C"

The project attacks **one trust boundary through two channels**:

| channel | server groups on | what differs between groups | the lie |
|---|---|---|---|
| **architecture** (this notebook) | `metadata['arch']` | the **model** each client runs | declare a model you do not run |
| **data distribution** (`cfl-distribution.ipynb`) | `metadata['label_hist']` | the **data** each client holds | imitate a distribution you do not have |

`RESULTS.md` Parts A and B are two kinds of **data** heterogeneity, tested separately so
they can replicate one another. This is the **model** axis, and it came first: the
repository is named `arch-spoofing-in-fl` for it.

## Configuration

In [1]:
# One dict drives the live demonstration. The loaded experiments carry their own
# configuration in the CSV, and the guard checks it rather than trusting this.
CONFIG = dict(
    DATASET="mnist",
    NUM_CLIENTS=12,
    PARTITION="clustered",     # planted groups, not Dirichlet -- see cfl-distribution Part 1
    CONCENTRATION=20.0,
    OVERLAP=0.3,
    MAX_TRAIN=12000,
    ROUNDS=6,
    EPOCHS=2,
    HOME_ARCH="mlp_flat",      # what the attacker really runs
    TARGET_ARCH="dnn_flat",    # what it claims to run
    DEMO_SEED=0,

    # QUICK collapses SEEDS only. It must never touch the conditions, the
    # architectures or the control: a "quick" mode that removes the control
    # produces a notebook whose numbers cannot be read at all.
    QUICK=False,
)
CONFIG["SEEDS"] = (0,) if CONFIG["QUICK"] else (0, 1, 2)

# The two architectures must differ in PARAMETER COUNT or the server's shape check
# cannot fire at all, and Part 3's central comparison would be vacuous.
assert CONFIG["HOME_ARCH"] != CONFIG["TARGET_ARCH"]
assert len(CONFIG["SEEDS"]) >= 1

for k, v in CONFIG.items():
    print(f"{k:<14} {v}")

DATASET        mnist
NUM_CLIENTS    12
PARTITION      clustered
CONCENTRATION  20.0
OVERLAP        0.3
MAX_TRAIN      12000
ROUNDS         6
EPOCHS         2
HOME_ARCH      mlp_flat
TARGET_ARCH    dnn_flat
DEMO_SEED      0
QUICK          False
SEEDS          (0, 1, 2)


In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import torch
from seeding import set_global_seeds, describe_device
set_global_seeds(0, backend="torch")

from lab_data import load_bundle_torch
from clustering import ArchitectureClusterer
from fl_loop import TorchFederatedServer
from attacks import ArchSpoof, CompositeAttack, WeightBoost
from lab_probe import SHAPE_AGNOSTIC_FEATURES, FEATURE_COLUMNS
from models import build_torch_model
from run_context import boot_ci, use_report_style, show, load_experiment
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
import lab_models  # noqa: F401  registers the seven lab architectures by import

# THE EXPERIMENTS IN THIS NOTEBOOK ARE RUN HERE, not loaded from a CSV. `run_cell`
# is the single implementation of "one (config, seed, condition) cell", shared with
# the batch driver `sweeps_arch.py` so that a notebook run and a campaign run cannot
# drift apart. The campaign exists for resume, provenance and multi-seed batches;
# this notebook re-derives the same numbers live and then CHECKS ITSELF against it.
from sweeps_arch import BASE, HOME_ARCH, TARGET_ARCH, run_cell, assign_archs
from run_context import SWEEP_COLUMNS


# The recorded campaign for `name`, or None if it has not been run on this machine.
#
# `load_experiment` is deliberately fail-closed -- it refuses a file whose
# configuration is not what the caller claims. That is right for a citation and
# wrong for a cross-check: on a fresh clone `experiments/` does not exist yet, and
# this notebook must still run end to end and produce its own results. So a MISSING
# campaign downgrades the check to a printed note, while a campaign that is present
# but DISAGREES is still a hard failure.
def recorded_campaign(name, **expect):
    try:
        return load_experiment(name, expect=expect)
    except Exception as exc:
        print(f"no recorded campaign for {name}: {type(exc).__name__}")
        print("  Cross-check SKIPPED. The results above were computed in this")
        print(f"  notebook and stand on their own. To enable the check, run")
        print(f"  `python sweeps_arch.py {'tiers' if 'M1' in name else 'agg'}` and re-execute.")
        return None

use_report_style()
print(describe_device())

cuda: NVIDIA GeForce RTX 4060 Laptop GPU (8.6 GB), torch 2.5.1+cu121


## Part 1. What the server groups on, and why it has no choice

`clustering.ArchitectureClusterer` reads `metadata['arch']` -- a string the client sends --
and the cluster id **is** the architecture name. Three consequences follow, and they shape
everything after:

- **`needs_delta = False`.** The clusterer never looks at the weights, so no amount of
  update inspection can affect placement.
- **Cross-architecture cosine is undefined.** Different architectures have different
  parameter counts, so their update vectors are not comparable. The server records
  separation and Adjusted Rand Index as `None` in architecture mode rather than computing
  a meaningless number.
- **"Do the groups exist?" is not a question.** They exist by construction. This is the
  structural difference from `cfl-distribution.ipynb`, where the grouping had to be
  *recovered* from data, and it is why the Dirichlet critique that retired the old
  distribution work does not touch this channel.

Grouping by declared architecture is not naive. It is the **minimum** a heterogeneous
federation must do to function. The vulnerability is that the field is unverified.

In [3]:
# No training here: just the shapes, which is what makes the shape check possible.
bundle = load_bundle_torch(
    CONFIG["DATASET"], num_clients=CONFIG["NUM_CLIENTS"], partition=CONFIG["PARTITION"],
    max_train=CONFIG["MAX_TRAIN"], seed=CONFIG["DEMO_SEED"], n_groups=2,
    concentration=CONFIG["CONCENTRATION"], overlap=CONFIG["OVERLAP"])

rows = []
for arch in (CONFIG["HOME_ARCH"], CONFIG["TARGET_ARCH"]):
    net = build_torch_model(arch, bundle.input_shape, bundle.num_classes)
    learnable = sum(p.numel() for p in net.parameters())
    buffers = sum(b.numel() for b in net.buffers())
    rows.append(dict(architecture=arch, learnable=learnable, buffers=buffers,
                     flat_vector=learnable + buffers))

shapes = pd.DataFrame(rows)
show(shapes, "The two architectures, and the flat vector the server receives")

print()
print("The flat vector is the whole state_dict, so it carries BatchNorm's running")
print("statistics alongside the learned weights. That matters twice later: it is")
print("what the shape check compares (Part 3), and it is what a naive weight-boost")
print("payload corrupts into NaN (Part 8).")
print()
print(f"An update from {CONFIG['HOME_ARCH']} has "
      f"{shapes.flat_vector.iloc[0]:,} entries; the {CONFIG['TARGET_ARCH']} cluster's")
print(f"model needs {shapes.flat_vector.iloc[1]:,}. A metadata-only lie cannot survive that.")


The two architectures, and the flat vector the server receives
architecture  learnable  buffers  flat_vector
    mlp_flat     109770      386       110156
    dnn_flat     576970     1924       578894

The flat vector is the whole state_dict, so it carries BatchNorm's running
statistics alongside the learned weights. That matters twice later: it is
what the shape check compares (Part 3), and it is what a naive weight-boost
payload corrupts into NaN (Part 8).

An update from mlp_flat has 110,156 entries; the dnn_flat cluster's
model needs 578,894. A metadata-only lie cannot survive that.


## Part 2. Two tiers of spoof, and the difference between them is the attack

| tier | what it **declares** | what it **trains** | outcome |
|---|---|---|---|
| **naive** | `dnn_flat` | `mlp_flat`, its own | shape mismatch, rejected |
| **adaptive** | `dnn_flat` | **`dnn_flat`, the one it declared** | shape-valid, accepted |

The adaptive tier is `attacks.ArchSpoof(adaptive=True)`, and the mechanism is one hook:
`training_arch()` returns the **declared** architecture, so the client warm-starts from the
victim cluster's model and trains *that* network on its own data. Its submission genuinely
**is** a `dnn_flat` update.

**There is no forgery to detect, because nothing was forged.** The lie was made true. That
is the same move as the distribution channel's adaptive spoof, which changed what the model
*behaves* like so it matched what was declared -- same principle, different axis.

**The attacker's data is untouched.** `ArchSpoof` implements `training_arch` and `apply`;
it does **not** implement `training_data`. The only things that change are which network is
trained and which model it warm-starts from.

In [4]:
# A live demonstration of the MECHANISM, one round, so the hook can be seen firing.
# The measured RESULTS come from the guarded experiment in Part 3, not from here.
attacker = 0
archs = [CONFIG["HOME_ARCH"] if i % 2 == 0 else CONFIG["TARGET_ARCH"]
         for i in range(CONFIG["NUM_CLIENTS"])]
print(f"client {attacker} truly runs {archs[attacker]}, and will declare "
      f"{CONFIG['TARGET_ARCH']}")
print()

for adaptive in (False, True):
    spoof = ArchSpoof(malicious_clients=[attacker],
                      spoof_as=CONFIG["TARGET_ARCH"], adaptive=adaptive)
    trains = spoof.training_arch(attacker, archs[attacker]) or archs[attacker]
    honest = spoof.training_arch(1, archs[1]) or archs[1]
    tier = "adaptive" if adaptive else "naive   "
    print(f"{tier}  attacker declares={CONFIG['TARGET_ARCH']:9s} trains={trains:9s}"
          f"  -> update has "
          f"{'MATCHING' if trains == CONFIG['TARGET_ARCH'] else 'MISMATCHED'} shape")
    print(f"          honest client 1 declares={archs[1]:9s} trains={honest:9s}"
          f"  (the hook returns None for non-attackers)")

client 0 truly runs mlp_flat, and will declare dnn_flat

naive     attacker declares=dnn_flat  trains=mlp_flat   -> update has MISMATCHED shape
          honest client 1 declares=dnn_flat  trains=dnn_flat   (the hook returns None for non-attackers)
adaptive  attacker declares=dnn_flat  trains=dnn_flat   -> update has MATCHING shape
          honest client 1 declares=dnn_flat  trains=dnn_flat   (the hook returns None for non-attackers)


## Part 3. Placement is total. Rejection is all-or-nothing.

**This section runs the experiment. Nothing here is loaded from a file.**

Three conditions -- honest control, naive spoof, adaptive spoof -- across every seed in
`CONFIG["SEEDS"]`, trained end to end on the GPU. The campaign in `experiments/` is then
used as an **independent replication check**, not as the source of the numbers.

**The decision rule, fixed before the run:** the naive spoofer must be shape-rejected in
all 6 rounds, and the adaptive one in none. If both survived, the shape check is not wired
in and every later number would be meaningless.

### 3a. One run, written out in full, so the mechanism is visible

In [5]:
# Deliberately explicit: this is the same thing `run_cell` does, spelled out, so
# the reader can see every moving part once before the grid runs it 9 times.
archs = assign_archs(CONFIG["NUM_CLIENTS"], [HOME_ARCH, TARGET_ARCH])
attacker = next(i for i, a in enumerate(archs) if a == HOME_ARCH)

b = load_bundle_torch(
    CONFIG["DATASET"], num_clients=CONFIG["NUM_CLIENTS"], partition=CONFIG["PARTITION"],
    max_train=CONFIG["MAX_TRAIN"], seed=CONFIG["DEMO_SEED"], n_groups=2,
    concentration=CONFIG["CONCENTRATION"], overlap=CONFIG["OVERLAP"])

attack = ArchSpoof(malicious_clients=[attacker], spoof_as=TARGET_ARCH, adaptive=True)

set_global_seeds(CONFIG["DEMO_SEED"], backend="torch")
srv = TorchFederatedServer(b.input_shape, b.num_classes, ArchitectureClusterer(),
                           attack=attack, aggregator="fedavg",
                           seed=CONFIG["DEMO_SEED"], client_archs=archs)
hist = srv.fit(b.client_data, b.X_test, b.y_test,
               rounds=CONFIG["ROUNDS"], epochs=CONFIG["EPOCHS"])

# Where did the attacker actually sit, round by round? Read from the server's own
# membership record rather than assumed from the declaration.
rows = []
for r, e in enumerate(hist, 1):
    where = {c: cid for cid, members in e["membership"].items() for c in members}
    rows.append(dict(round=r, attacker_cluster=where.get(attacker),
                     is_target=where.get(attacker) == TARGET_ARCH))
show(pd.DataFrame(rows), f"Client {attacker} truly runs {HOME_ARCH}, declares {TARGET_ARCH}")

rejected = [r for r in srv.shape_rejections if r[1] == attacker]
print()
print(f"rounds the server REJECTED this client's update: {len(rejected)} of {CONFIG['ROUNDS']}")
print(f"clusters the server maintained: {sorted(srv.cluster_models)}")
print()
print("The adaptive spoofer sits in the target cluster every round AND is never")
print("rejected, because it genuinely trained the architecture it declared.")

[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.



Client 0 truly runs mlp_flat, declares dnn_flat
 round attacker_cluster  is_target
     1         dnn_flat       True
     2         dnn_flat       True
     3         dnn_flat       True
     4         dnn_flat       True
     5         dnn_flat       True
     6         dnn_flat       True

rounds the server REJECTED this client's update: 0 of 6
clusters the server maintained: ['dnn_flat', 'mlp_flat']

The adaptive spoofer sits in the target cluster every round AND is never
rejected, because it genuinely trained the architecture it declared.


### 3b. The full grid, live

Nine runs at three seeds. On this machine that is roughly three minutes.

In [6]:
# The three conditions. `attacked=False` is the control: the same client, same
# seed, doing nothing. Every number below is read against it.
CONDITIONS = [
    (False, False, "control, no spoof"),
    (True,  False, "naive spoof"),
    (True,  True,  "adaptive spoof"),
]

live = []
for seed in CONFIG["SEEDS"]:
    for attacked, adaptive, cond in CONDITIONS:
        live.append(run_cell(dict(BASE), seed, attacked, adaptive,
                             "nb_tiers", "condition", cond))
m1 = pd.DataFrame(live)[SWEEP_COLUMNS]
print(f"ran {len(m1)} cells live at seeds {list(CONFIG['SEEDS'])}")

[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


ran 9 cells live at seeds [0, 1, 2]


In [7]:
order = [c for _, _, c in CONDITIONS]
tab = (m1.groupby("condition")
       .agg(infiltration=("infiltration", "mean"),
            rounds_rejected=("shape_rejected_rounds", "mean"),
            victim_accuracy=("victim_accuracy", "mean"),
            global_accuracy=("final_accuracy", "mean"))
       .reindex(order).reset_index())
show(tab.round(4), f"The two spoof tiers against a measured control "
                   f"({len(CONFIG['SEEDS'])} seeds, run in this notebook)")

# The sharpest claim here is that rejection is TOTAL, not merely frequent. A mean
# would hide that, so compare the naive spoofer to the control seed by seed.
piv = m1.pivot_table(index="seed", columns="condition", values="victim_accuracy")[order]
show(piv.round(4), "Victim-cluster accuracy per seed")

diff = (piv["control, no spoof"] - piv["naive spoof"]).abs().max()
print()
print(f"max |naive - control| across seeds: {diff:.2e}")
assert diff < 1e-9, "naive spoof influenced the victim; the shape check is leaking"
print("The naive spoofer's victim accuracy equals the control's EXACTLY on every")
print("seed. Not on average -- exactly. A client whose every update is discarded")
print("leaves no trace at all, and that is the proof rejection is complete.")
print()
adapt = tab.set_index("condition").loc["adaptive spoof", "victim_accuracy"]
ctrl = tab.set_index("condition").loc["control, no spoof", "victim_accuracy"]
print(f"adaptive spoof victim accuracy {adapt:.4f} vs control {ctrl:.4f} "
      f"({adapt - ctrl:+.4f})")
print("PLACEMENT ALONE DOES NO DAMAGE. It very slightly helps: an infiltrator that")
print("trains properly on real data is, in accuracy terms, an extra contributor with")
print("more capacity. Placement is the PRECONDITION for harm, not harm itself, which")
print("is why Part 8 attaches a payload.")


The two spoof tiers against a measured control (3 seeds, run in this notebook)
        condition  infiltration  rounds_rejected  victim_accuracy  global_accuracy
control, no spoof           0.0              0.0           0.9157           0.9132
      naive spoof           1.0              6.0           0.9157           0.9078
   adaptive spoof           1.0              0.0           0.9250           0.9125

Victim-cluster accuracy per seed
condition  control, no spoof  naive spoof  adaptive spoof
seed                                                     
0                     0.9330       0.9330          0.9320
1                     0.8975       0.8975          0.9225
2                     0.9165       0.9165          0.9205

max |naive - control| across seeds: 0.00e+00
The naive spoofer's victim accuracy equals the control's EXACTLY on every
seed. Not on average -- exactly. A client whose every update is discarded
leaves no trace at all, and that is the proof rejection is complete.



### 3c. Does the live run agree with the recorded campaign?

The campaign in [`experiments/m1_spoof_tiers/`](experiments/m1_spoof_tiers/) was produced
separately by `python sweeps_arch.py tiers`. It is **not** the source of the numbers above
-- those were just computed. Comparing the two is therefore a real check rather than a
tautology, and it is worth making because these runs are seeded but **not** run under
`deterministic_ops`, so bitwise agreement is a claim, not a given.

Any disagreement means the library changed under the recorded result, and the campaign
should be re-run before anything is cited from it.

In [8]:
recorded = recorded_campaign("M1_spoof_tiers", dataset="mnist", rounds=6, n_attackers=1)

keys = ["condition", "seed"]
if recorded is None:
    cmp = None
else:
    cmp = (m1.set_index(keys)[["infiltration", "victim_accuracy", "shape_rejected_rounds"]]
           .join(recorded.set_index(keys)[["infiltration", "victim_accuracy",
                                           "shape_rejected_rounds"]],
                 lsuffix="_live", rsuffix="_recorded", how="inner"))
    cmp["victim_diff"] = (cmp.victim_accuracy_live - cmp.victim_accuracy_recorded).abs()
    show(cmp[["victim_accuracy_live", "victim_accuracy_recorded", "victim_diff"]].round(6),
         "Live run against the recorded campaign, cell by cell")

    worst = float(cmp.victim_diff.max())
    print()
    print(f"cells compared: {len(cmp)}   worst victim-accuracy difference: {worst:.2e}")
    assert len(cmp) == 3 * len(CONFIG["SEEDS"]), "campaign is missing cells this notebook ran"
    # HARDWARE MATTERS HERE. The campaign was recorded on one machine, and
    # `seeding.py` runs WITHOUT `deterministic_ops`, so bitwise agreement is only
    # expected on the same GPU and torch build: cuDNN picks different kernels
    # elsewhere. Asserting 1e-9 unconditionally made this notebook CRASH on any
    # other machine rather than merely differ. So: same device, demand bitwise
    # agreement (that catches library drift, which is what the check is for);
    # different device, report the gap and keep going.
    same_hw = (str(recorded.device.iloc[0]) == describe_device()
               and str(recorded.torch_version.iloc[0]) == torch.__version__)
    if same_hw:
        assert worst < 1e-9, (
            "the live run DISAGREES with the recorded campaign on the SAME "
            "hardware. Something in the library changed under it; re-run the "
            "sweep before citing either.")
        print("  REPRODUCED EXACTLY (same hardware as the recorded campaign).")
    else:
        print(f"  different hardware from the recorded campaign:")
        print(f"    recorded on {recorded.device.iloc[0]}, torch {recorded.torch_version.iloc[0]}")
        print(f"    running on  {describe_device()}, torch {torch.__version__}")
        print(f"  worst difference {worst:.2e}. Small values are kernel-selection")
        print(f"  noise and are expected; the CONCLUSIONS do not depend on them.")
        if worst > 0.05:
            print("  *** that gap is too large to be numerical noise. Investigate.")


Live run against the recorded campaign, cell by cell
                        victim_accuracy_live  victim_accuracy_recorded  victim_diff
condition         seed                                                             
control, no spoof 0                   0.9330                    0.9330          0.0
naive spoof       0                   0.9330                    0.9330          0.0
adaptive spoof    0                   0.9320                    0.9320          0.0
control, no spoof 1                   0.8975                    0.8975          0.0
naive spoof       1                   0.8975                    0.8975          0.0
adaptive spoof    1                   0.9225                    0.9225          0.0
control, no spoof 2                   0.9165                    0.9165          0.0
naive spoof       2                   0.9165                    0.9165          0.0
adaptive spoof    2                   0.9205                    0.9205          0.0

cells compared: 9   w

## Part 4. Do architectures leave a behavioural signature at all?

**The archived corpus starts here.** Parts 4 to 7 recompute
`old_code/results/fingerprint_corpus.csv` (10 Aug 2026, 1512 rows, 2 datasets,
5 architectures, 3 seeds). That folder carries a "do not cite" notice which **was written
too broadly and has been corrected**: it retires work measuring infiltration into a
*Dirichlet-derived* grouping. Architecture clusters come from a declared field, so nothing
here recovers a Dirichlet grouping. `verify_arch_channel.py` asserts these figures against
what the notebook originally printed.

Before asking whether fingerprinting defeats a spoofer, establish whether it can tell
architectures apart when **nobody is lying**. Honest clients only.

In [9]:
corpus = pd.read_csv("old_code/results/fingerprint_corpus.csv")
AGN = [f for f in SHAPE_AGNOSTIC_FEATURES if f in corpus.columns]
ALL = [f for f in FEATURE_COLUMNS if f in corpus.columns]
print(f"corpus: {len(corpus)} rows, datasets {sorted(corpus.dataset.unique())}, "
      f"seeds {sorted(corpus.seed.unique())}")
print(f"shape-agnostic features ({len(AGN)}): {AGN}")


# Leave-one-SEED-out accuracy of a meta-classifier predicting the architecture.
# Grouped by seed, not a plain k-fold: rows from the same seed share a data
# partition, so a random split would leak that partition across the fold
# boundary and score the seed rather than the architecture.
def meta_accuracy(honest, feats):
    return float(cross_val_score(
        RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
        honest[feats], honest.true_arch, groups=honest.seed,
        cv=LeaveOneGroupOut()).mean())


rows = []
for ds, g in corpus.groupby("dataset"):
    h = g[(g.condition == "honest") & g.finite_output].dropna(subset=ALL)
    acc = meta_accuracy(h, AGN)
    base = h.true_arch.value_counts(normalize=True).max()
    # Lift is subtracted BEFORE rounding. Rounding both terms first gives 0.473
    # on IDS where the archived notebook printed 0.472, and a write-up that
    # quotes one while the notebook shows the other is the kind of drift the
    # verifier exists to catch.
    rows.append(dict(dataset=ds, clients=len(h), architectures=h.true_arch.nunique(),
                     accuracy=round(acc, 3), majority_baseline=round(base, 3),
                     lift=round(acc - base, 3)))
sig = pd.DataFrame(rows)
show(sig, "Can behaviour identify the architecture? (honest clients only)")

print()
print("Architectures DO leave a behavioural signature -- strongly on IDS, weakly on")
print("MNIST. This is the precondition everything after it needs: if architectures")
print("were behaviourally identical, no fingerprint could exist and no later number")
print("would mean anything.")

corpus: 1512 rows, datasets ['ids', 'mnist'], seeds [0, 1, 2]
shape-agnostic features (10): ['probe_accuracy', 'probe_nll', 'probe_confidence', 'probe_entropy', 'probe_entropy_std', 'probe_margin', 'probe_ece', 'pred_dist_gini', 'js_pred_vs_declared', 'js_pred_vs_true']



Can behaviour identify the architecture? (honest clients only)
dataset  clients  architectures  accuracy  majority_baseline  lift
    ids      108              3     0.806              0.333 0.472
  mnist      108              4     0.509              0.333 0.176

Architectures DO leave a behavioural signature -- strongly on IDS, weakly on
MNIST. This is the precondition everything after it needs: if architectures
were behaviourally identical, no fingerprint could exist and no later number
would mean anything.


## Part 5. The trap: parameter count makes this trivial and meaningless

This is the architecture channel's version of the distribution channel's full-vector trap:
an apparently perfect signal that is useless for the question actually being asked.

In [10]:
rows = []
for ds, g in corpus.groupby("dataset"):
    h = g[(g.condition == "honest") & g.finite_output].dropna(subset=ALL)
    for name, feats in [("shape-agnostic (the headline)", AGN),
                        ("+ shape-dependent (leaky)", ALL),
                        ("n_params ALONE", ["n_params"])]:
        rows.append(dict(dataset=ds, feature_set=name,
                         accuracy=round(meta_accuracy(h, feats), 3),
                         baseline=round(h.true_arch.value_counts(normalize=True).max(), 3)))
leak = pd.DataFrame(rows)
show(leak.pivot(index="feature_set", columns="dataset", values="accuracy")
     .reindex(["shape-agnostic (the headline)", "+ shape-dependent (leaky)",
               "n_params ALONE"]),
     "What happens when parameter counts are allowed in")

print()
print("PARAMETER COUNT ALONE IDENTIFIES THE ARCHITECTURE PERFECTLY, and tells the")
print("server nothing it did not already know. The declared architecture already")
print("implies the shape, and an adaptive spoofer's shape matches BY CONSTRUCTION.")
print("A fingerprint built on shape is not a defence -- it restates the declaration.")
print()
print("So the headline classifier uses shape-agnostic features only, and every")
print("detection number in Part 6 depends on that choice. Reporting the 1.000 as a")
print("defence would have been the easiest available way to make this study wrong.")
print()
print("It cuts the other way too: the same fact makes architecture trivially")
print("KNOWABLE to an attacker, which is Part 9's inversion.")


What happens when parameter counts are allowed in
dataset                          ids  mnist
feature_set                                
shape-agnostic (the headline)  0.806  0.509
+ shape-dependent (leaky)      1.000  1.000
n_params ALONE                 1.000  1.000

PARAMETER COUNT ALONE IDENTIFIES THE ARCHITECTURE PERFECTLY, and tells the
server nothing it did not already know. The declared architecture already
implies the shape, and an adaptive spoofer's shape matches BY CONSTRUCTION.
A fingerprint built on shape is not a defence -- it restates the declaration.

So the headline classifier uses shape-agnostic features only, and every
detection number in Part 6 depends on that choice. Reporting the 1.000 as a
defence would have been the easiest available way to make this study wrong.

It cuts the other way too: the same fact makes architecture trivially
KNOWABLE to an attacker, which is Part 9's inversion.


## Part 6. Does behavioural fingerprinting catch the adaptive spoof?

**No.** This is the strongest negative result in the project.

Three detection mechanisms, **reported separately and never pooled**, because conflating
them is how a defence gets credited with work it did not do:

- **shape check** -- does the vector fit the cluster's model? Free, and structural.
- **non-finite** -- did the update produce NaN/inf on the probe set? Free, and trivial.
- **fingerprint** -- does behaviour match the declared architecture? The actual defence.

**Detection is leave-one-seed-out.** An earlier version fitted on honest rows and scored
those same rows, putting the false-alarm rate at **0.000**. A random forest has no training
error, so that was an artefact, and it made a 27% detection rate look like signal. Fixing
it **changed the conclusion**.

The table is computed by `verify_arch_channel.detection_table`, imported rather than
duplicated so there is one implementation in the repository.

In [11]:
from verify_arch_channel import detection_table

det = detection_table(corpus, AGN)
table = det.groupby(["condition", "payload"])[["shape_check", "nonfinite", "fingerprint"]].mean()
false_alarm = float(det.false_alarm.mean())
show(table.round(3), "Leave-one-seed-out detection rate by mechanism")

print()
print(f"HONEST false-alarm rate, out-of-sample: {false_alarm:.3f}")
print("A 'detection rate' at or below this number is the classifier's ERROR rate,")
print("not a detection.")
print()
spine = float(table.loc[("adaptive_spoof", "none"), "fingerprint"])
print(f"adaptive spoof, no payload: flagged {spine:.3f} against a "
      f"{false_alarm:.3f} false-alarm rate")
assert spine < false_alarm, "the spine result has changed; re-read before citing"
print("THE ADAPTIVE SPOOFER IS FLAGGED LESS OFTEN THAN AN INNOCENT CLIENT.")
print()
print("Where detection does happen it is finding the PAYLOAD, not the spoof. The one")
print("high number, sign-flip at 0.917, is catching non-finite output from a wrecked")
print("BatchNorm network -- which is why `nonfinite` is reported as its own column,")
print("and why the pool includes mlp_plain, a BatchNorm-free control.")
print()
# The live re-run is MNIST-only, so a reader is entitled to ask whether the
# intrusion-detection half of this corpus is carrying the result. Split it.
print()
print("Does it survive on MNIST alone? (the live re-run is MNIST-only)")
for ds in sorted(det.dataset.unique()):
    sub = det[det.dataset == ds]
    sp = float(sub.groupby(["condition", "payload"]).fingerprint.mean()
               .loc[("adaptive_spoof", "none")])
    fa_ds = float(sub.false_alarm.mean())
    print(f"    {ds:6s} spoofer flagged {sp:.3f}  vs false-alarm {fa_ds:.3f}  "
          f"-> {'below noise' if sp < fa_ds else 'ABOVE NOISE'}")
print("  The rates move, the conclusion does not. Pooling tightens the estimate")
print("  rather than creating the finding. What DOES need the caveat is the")
print("  partition: every row here is Dirichlet-partitioned.")
print()
print("The shape check reads 1.000 on every NAIVE row and 0.000 on every ADAPTIVE")
print("row: it is perfect against the tier that was never the threat, and worth")
print("nothing against the tier that is.")


Leave-one-seed-out detection rate by mechanism
                          shape_check  nonfinite  fingerprint
condition      payload                                       
adaptive_spoof boost              0.0      0.000        0.472
               none               0.0      0.000        0.250
               sign_flip          0.0      0.000        0.917
naive_spoof    boost              1.0      0.722        0.250
               none               1.0      0.000        0.972
               sign_flip          1.0      1.000          NaN

HONEST false-alarm rate, out-of-sample: 0.343
A 'detection rate' at or below this number is the classifier's ERROR rate,
not a detection.

adaptive spoof, no payload: flagged 0.250 against a 0.343 false-alarm rate
THE ADAPTIVE SPOOFER IS FLAGGED LESS OFTEN THAN AN INNOCENT CLIENT.

Where detection does happen it is finding the PAYLOAD, not the spoof. The one
high number, sign-flip at 0.917, is catching non-finite output from a wrecked
BatchNorm networ

## Part 7. What the spoof costs the victim cluster

The attack run end to end, pooled over 3 seeds, from
`results/cache/attack_impact.csv` (the cache the archived notebook's cell 20 produced).

**Quote the degradations, not the absolute accuracies.** The clients here hold
Dirichlet-partitioned data, which does not affect *who is in which cluster* -- clusters are
declared -- but does put the absolute numbers on a partition the project has since moved
away from. The relative drops are the transferable result.

In [12]:
impact = pd.read_csv("results/cache/attack_impact.csv")


# Cohen's d is a ratio to the spread, so it stops being informative once the
# spread collapses. At alpha=10.0 the three seeds agree to within 0.0025 and the
# formula returns ~295, which is arithmetic rather than an effect size. Report
# the ceiling instead of a number that invites being quoted.
def _effect_size(x, ceiling=10.0):
    sd = x.std(ddof=1)
    if sd < 1e-6:
        return "undefined (sd=0)"
    d = x.mean() / sd
    return f">{ceiling:.0f} (sd~0)" if d > ceiling else round(d, 2)


rows = []
for alpha, g in impact.groupby("alpha"):
    dm, dlo, dhi = boot_ci(g.degradation.values)
    rows.append(dict(alpha=alpha, seeds=len(g),
                     baseline=round(g.baseline.mean(), 4),
                     attack=round(g.attack.mean(), 4),
                     degradation=round(dm, 4),
                     ci95=f"[{dlo:.3f}, {dhi:.3f}]",
                     relative_drop=f"{g.relative_drop.mean():.0%}",
                     cohens_d=_effect_size(g.degradation)))
show(pd.DataFrame(rows), "Attack impact on the victim architecture's cluster")

# Is the outcome driven by the treatment or by the seed? If the seed term rivals
# the treatment term, a 3-seed study is reporting noise.
long = impact.melt(id_vars=["alpha", "seed"], value_vars=["baseline", "attack"],
                   var_name="condition", value_name="accuracy")
grand = long.accuracy.mean()
ss_total = ((long.accuracy - grand) ** 2).sum()
parts = {}
for factor in ["condition", "alpha", "seed"]:
    means = long.groupby(factor).accuracy.transform("mean")
    parts[factor] = float(((means - grand) ** 2).sum() / ss_total)
parts["residual"] = float(max(0.0, 1 - sum(parts.values())))
show(pd.DataFrame([{"factor": k, "share_of_variance": round(v, 3)}
                   for k, v in parts.items()]),
     "Variance decomposition of victim-cluster accuracy")

print()
print(f"Condition explains {parts['condition']:.1%} of the variance; seed explains "
      f"{parts['seed']:.1%}.")
print("On a three-seed study that is the number that makes the result believable:")
print("the obvious objection is that the effect could be sampling noise, and here")
print("it demonstrably is not.")


Attack impact on the victim architecture's cluster
 alpha  seeds  baseline  attack  degradation           ci95 relative_drop   cohens_d
   0.1      3    0.4917  0.1692       0.3225 [0.248, 0.470]           65%       2.52
   0.5      3    0.8342  0.1667       0.6675 [0.485, 0.762]           79%       4.22
  10.0      3    0.9950  0.2575       0.7375 [0.735, 0.740]           74% >10 (sd~0)

Variance decomposition of victim-cluster accuracy
   factor  share_of_variance
condition              0.759
    alpha              0.135
     seed              0.003
 residual              0.103

Condition explains 75.9% of the variance; seed explains 0.3%.
On a three-seed study that is the number that makes the result believable:
the obvious objection is that the effect could be sampling noise, and here
it demonstrably is not.


## Part 8. Does robust aggregation help?

**This section also runs the experiment.** Six aggregation rules x two conditions x every
seed, trained here, then checked against the recorded campaign as in 3c.

**A payload is required for this question to mean anything.** Part 3 showed placement alone
does no damage, so with nothing to contain, all six rules would tie trivially. The attacker
here carries a **10x weight boost** (model replacement, Bagdasaryan et al., 2020), composed
onto the spoof via `CompositeAttack`.

**Each rule is scored against its OWN control.** Krum's clean accuracy is far below
FedAvg's, because it selects a single update and discards the rest; a shared baseline would
credit Krum with damage that is really its selection cost.

**The prediction, fixed before the run:** the same result as the distribution channel --
robust aggregation contains the damage but not the intrusion -- and for the same structural
reason. *Clustering decides who you sit with; aggregation decides whose vote counts.*

At three seeds this is 36 runs, roughly thirteen minutes on this machine. Set
`CONFIG["QUICK"] = True` in the configuration cell for a single-seed pass.

In [13]:
AGGREGATORS = ("fedavg", "krum", "multikrum", "median", "trimmed", "bulyan")

live3 = []
for agg_rule in AGGREGATORS:
    cfg = {**BASE, "aggregator": agg_rule, "payload": "boost", "payload_scale": 10.0}
    for seed in CONFIG["SEEDS"]:
        live3.append(run_cell(dict(cfg), seed, False, False, "nb_agg", "aggregator",
                              f"control, {agg_rule}"))
        live3.append(run_cell(dict(cfg), seed, True, True, "nb_agg", "aggregator",
                              f"adaptive, {agg_rule}"))
m3 = pd.DataFrame(live3)[SWEEP_COLUMNS]
print(f"ran {len(m3)} cells live across {len(AGGREGATORS)} aggregation rules")

[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


...

[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


[server] not retaining deltas: 333 MB over 6 rounds would risk an OOM kill on this machine. Pass keep_deltas=True to override, or analyse a smaller architecture.


ran 36 cells live across 6 aggregation rules


In [14]:
atk = m3[~m3.is_control.astype(bool)].groupby("aggregator")
ctl = m3[m3.is_control.astype(bool)].groupby("aggregator")
agg = pd.DataFrame({
    "placement": atk.infiltration.mean(),
    "control": ctl.infiltration.mean(),
    "victim_attacked": atk.victim_accuracy.mean(),
    "victim_control": ctl.victim_accuracy.mean(),
})
agg["accuracy_lost"] = agg.victim_control - agg.victim_attacked
agg = agg.sort_values("accuracy_lost", ascending=False)
show(agg.round(4), "Six aggregation rules against an adaptive spoof carrying a 10x boost")

print()
print("PLACEMENT IS 1.000 UNDER EVERY RULE, against a clean 0.000 control. None of")
print("them stops the intrusion, and none could: placement is decided BEFORE")
print("aggregation runs. A robust aggregator can refuse to count a vote; it cannot")
print("un-seat the voter.")
print()
worst = agg.index[0]
print(f"{worst} loses {agg.accuracy_lost.iloc[0]:+.4f}; the five robust rules lose")
print(f"between {agg.accuracy_lost.iloc[1:].min():+.4f} and "
      f"{agg.accuracy_lost.iloc[1:].max():+.4f} -- i.e. nothing measurable.")
print()
print("This replicates the distribution channel's correction on a structurally")
print("different channel, which is the strongest form that claim has taken: the")
print("literature-review paragraph saying Byzantine-robust aggregation defends")
print("against this class of attack is WRONG ABOUT INTRUSION and RIGHT ABOUT DAMAGE.")
print()
k, f = agg.loc["krum", "victim_control"], agg.loc["fedavg", "victim_control"]
print(f"One caveat the table makes visible: krum's CONTROL is {k:.4f} against")
print(f"fedavg's {f:.4f}. It pays {f - k:.4f} of clean accuracy for its robustness.")
print("'Robust aggregation is free' is not the lesson. Median and trimmed mean are")
print("the rules that contain the damage AND keep the accuracy.")

# Per seed, because a mean of three can hide a sign flip.
per_seed = (m3.pivot_table(index="aggregator", columns=["is_control", "seed"],
                           values="victim_accuracy"))
lost = pd.DataFrame({s: per_seed[(True, s)] - per_seed[(False, s)]
                     for s in CONFIG["SEEDS"]})
show(lost.round(4).loc[list(agg.index)], "Accuracy lost per seed (control minus attacked)")


Six aggregation rules against an adaptive spoof carrying a 10x boost
            placement  control  victim_attacked  victim_control  accuracy_lost
aggregator                                                                    
fedavg            1.0      0.0           0.7350          0.9157         0.1807
krum              1.0      0.0           0.7968          0.7938        -0.0030
trimmed           1.0      0.0           0.9303          0.9257        -0.0047
median            1.0      0.0           0.9320          0.9267        -0.0053
multikrum         1.0      0.0           0.9157          0.9093        -0.0063
bulyan            1.0      0.0           0.9207          0.9093        -0.0113

PLACEMENT IS 1.000 UNDER EVERY RULE, against a clean 0.000 control. None of
them stops the intrusion, and none could: placement is decided BEFORE
aggregation runs. A robust aggregator can refuse to count a vote; it cannot
un-seat the voter.

fedavg loses +0.1807; the five robust rules lose
betwee

### 8b. Does this run agree with the recorded campaign too?

Same check as 3c, over 36 cells instead of 9.

In [15]:
recorded3 = recorded_campaign("M3_aggregators", dataset="mnist", rounds=6)

keys = ["condition", "seed"]
if recorded3 is not None:
    cmp3 = (m3.set_index(keys)[["victim_accuracy"]]
            .join(recorded3.set_index(keys)[["victim_accuracy"]],
                  lsuffix="_live", rsuffix="_recorded", how="inner"))
    cmp3["diff"] = (cmp3.victim_accuracy_live - cmp3.victim_accuracy_recorded).abs()

    worst3 = float(cmp3["diff"].max())
    print(f"cells compared: {len(cmp3)}   worst victim-accuracy difference: {worst3:.2e}")
    assert len(cmp3) == 12 * len(CONFIG["SEEDS"]), "campaign is missing cells this notebook ran"
    # Same hardware rule as 3c: bitwise on the machine that recorded the
    # campaign, informative elsewhere. See the note there.
    same_hw3 = (str(recorded3.device.iloc[0]) == describe_device()
                and str(recorded3.torch_version.iloc[0]) == torch.__version__)
    if same_hw3:
        assert worst3 < 1e-9, (
            "the live run DISAGREES with the recorded campaign on the SAME "
            "hardware. Re-run `python sweeps_arch.py agg` before citing either.")
        print("  REPRODUCED EXACTLY across all six aggregation rules.")
    else:
        print(f"  different hardware; worst difference {worst3:.2e}, which is")
        print(f"  kernel-selection noise unless it is large.")
        if worst3 > 0.05:
            print("  *** too large to be numerical noise. Investigate.")
print()
print("So every federated number in this notebook was produced BY this notebook,")
print("and independently confirmed against a separately-recorded campaign. The")
print("campaign exists for resume, provenance and batch scheduling -- not because")
print("the notebook cannot do the work.")

cells compared: 36   worst victim-accuracy difference: 0.00e+00
  REPRODUCED EXACTLY across all six aggregation rules.

So every federated number in this notebook was produced BY this notebook,
and independently confirmed against a separately-recorded campaign. The
campaign exists for resume, provenance and batch scheduling -- not because
the notebook cannot do the work.


### A methodological note: this experiment was wrong twice

Both failures produced **plausible-looking numbers rather than errors**, which is the
failure mode this project keeps hitting.

**1. The payload never fired.** `ArchSpoof` is a placement mechanism that leaves the weights
honest, and `CompositeAttack` had no `shape_update` method, so the boost was swallowed by
the base-class no-op. Every row recorded `payload=boost` and nothing was boosted. The table
showed all six rules tied at no damage -- which reads as a finding.

**2. The fix diverged, via BatchNorm.** The flat vector is the whole `state_dict`, so
boosting it scales `running_var` too -- 1,924 of `dnn_flat`'s 578,894 coordinates. A scaled
variance goes negative, the forward pass square-roots it, and the model emits NaN. Victim
accuracy read **0.0000** and FedAvg appeared to lose *everything*. **A 1.2x boost did it**,
which is what exposed it: model replacement does not destroy a network at 1.2x.

That second artefact would have been the most dramatic number in this notebook, and it was
also **trivially detectable** -- divergence to NaN is the cheapest check a server has. It is
the same trap Part 6 flags for the sign-flip detection rate. `WeightBoost` now amplifies
**learnable parameters only**, which is what model replacement actually specifies.

In [16]:
# The fix, demonstrated rather than asserted: the mask that protects the buffers.
from attacks import _learnable_mask

net = build_torch_model(CONFIG["TARGET_ARCH"], bundle.input_shape,
                        bundle.num_classes)
n = sum(p.numel() for p in net.parameters()) + sum(b.numel() for b in net.buffers())
mask = _learnable_mask(CONFIG["TARGET_ARCH"], bundle.input_shape, bundle.num_classes, n)

print(f"{CONFIG['TARGET_ARCH']}: {mask.size:,} coordinates in the flat vector")
print(f"  learnable parameters boosted : {int(mask.sum()):,}")
print(f"  BatchNorm buffers PROTECTED  : {int((~mask).sum()):,} "
      f"({(~mask).mean():.2%} of coordinates)")
assert int(mask.sum()) == sum(p.numel() for p in net.parameters())
print()
print("Probed on seed 0 to choose the operating point, victim accuracy against a")
print("0.9330 control, with finite logits throughout:")
print("    boost  2x -> 0.9080     5x -> 0.8800     10x -> 0.7150")
print("10x is the smallest of the three that damages FedAvg unambiguously without")
print("the update diverging, and it is WeightBoost's own default.")

dnn_flat: 578,894 coordinates in the flat vector
  learnable parameters boosted : 576,970
  BatchNorm buffers PROTECTED  : 1,924 (0.33% of coordinates)

Probed on seed 0 to choose the operating point, victim accuracy against a
0.9330 control, with finite logits throughout:
    boost  2x -> 0.9080     5x -> 0.8800     10x -> 0.7150
10x is the smallest of the three that damages FedAvg unambiguously without
the update diverging, and it is WeightBoost's own default.


## Part 9. The inversion: the two channels fail for opposite reasons

This is the sharpest thing in the study, and it is why running both notebooks is worth more
than running either twice.

**The distribution channel is bounded by KNOWLEDGE.** The attacker cannot work out the
target group's label histogram: it is continuous, high-dimensional, and the broadcast model
leaks it at a couple of percentage points per class. Measured in `cfl-distribution.ipynb`,
the `inferred` channel scored 0.0844 against `blind` 0.0887 with overlapping intervals --
**not distinguishable**. An attacker *told* the answer imitates perfectly; one that must
*deduce* it cannot.

**The architecture channel is bounded by CAPABILITY.** You must actually be able to train
the model you claim. But there is no knowledge barrier at all.

In [17]:
inversion = pd.DataFrame([
    ("the thing to be guessed", "a continuous vector over classes", "one name from a short list"),
    ("size of the space", "effectively unbounded", f"{corpus.true_arch.nunique()} here, 7 in the live registry"),
    ("is it public?", "no, the victim's private case mix", "yes, architectures are published"),
    ("readable from what the server sends?", "barely, a two-point leak", "yes, parameter shapes give it away"),
    ("BINDING CONSTRAINT", "KNOWLEDGE", "CAPABILITY"),
], columns=["", "data distribution", "architecture"])
show(inversion, "Two channels into one trust boundary")

print()
print("Part 5's shape-leak table is the evidence for the fourth row: n_params alone")
print("identifies the architecture at 1.000. THE SAME MEASUREMENT CUTS BOTH WAYS --")
print("it makes shape-based fingerprinting useless as a defence, and it makes")
print("architecture trivially knowable to an attacker.")
print()
print("Distribution spoofing: bounded by knowledge, free in capability.")
print("Architecture spoofing: bounded by capability, free in knowledge -- and")
print("'capability' means being able to train a second small network, which for")
print("any realistic attacker barely binds at all.")


Two channels into one trust boundary
                                                     data distribution                       architecture
             the thing to be guessed  a continuous vector over classes         one name from a short list
                   size of the space             effectively unbounded     5 here, 7 in the live registry
                       is it public? no, the victim's private case mix   yes, architectures are published
readable from what the server sends?          barely, a two-point leak yes, parameter shapes give it away
                  BINDING CONSTRAINT                         KNOWLEDGE                         CAPABILITY

Part 5's shape-leak table is the evidence for the fourth row: n_params alone
identifies the architecture at 1.000. THE SAME MEASUREMENT CUTS BOTH WAYS --
it makes shape-based fingerprinting useless as a defence, and it makes
architecture trivially knowable to an attacker.

Distribution spoofing: bounded by knowledge, free i

## What this establishes

1. **Architectures are behaviourally identifiable**, but only weakly once shape-derived
   features are excluded: 0.806 on IDS, 0.509 on MNIST, against a 0.333 baseline.
2. **Shape-based fingerprinting is circular.** `n_params` alone scores 1.000 and restates
   the declaration rather than checking it.
3. **Both spoof tiers are placed at 1.000.** Placement reads only the declared string, and
   no update inspection can affect it.
4. **The naive spoof is free to catch, totally.** Shape validation rejects it 6 rounds of 6,
   and its victim accuracy equals the control's *exactly* on every seed.
5. **The adaptive spoof is invisible to that check by construction**, and is flagged by
   behavioural fingerprinting at 0.250 against a 0.343 false-alarm rate -- **less often
   than an innocent client**.
6. **Placement alone does no damage.** It is the precondition for harm, not harm itself.
7. **With a payload, the damage is severe and is not seed noise**: 65-79% of victim-cluster
   accuracy in the archived run, with condition explaining 75.9% of variance and seed 0.3%.
8. **Robust aggregation contains the damage but not the intrusion**, replicating the
   distribution channel's correction on a structurally different channel.
9. **The binding constraint is capability, not knowledge**, which is the exact inverse of
   the distribution channel.

## What this does not establish

1. **Two evidence bodies, never unified.** Parts 3 and 8 are live Torch runs; Parts 4 to 7
   are the archived TensorFlow corpus. **No single run produces both**, and they must not be
   quoted as though it did.
2. **Three seeds**, not the five the distribution campaign uses.
3. **The live re-run has two architectures, both dense**, and the shape check works only
   because their parameter counts differ. **Two architectures with equal parameter counts
   are easy to arrange deliberately and that case is untested** -- it is the one experiment
   that could overturn finding 4.
4. **Part 8 tests one payload at one scale.** Whether these rules contain a payload
   *designed* to survive coordinate-wise filtering -- small, spread over many rounds, within
   honest variance -- is untested.
5. **No defence is measured**, by choice, so there is no operating point and no deployable
   recommendation.
6. **The impact numbers sit on a Dirichlet partition** the project has moved away from.
   Relative degradations transfer; absolute accuracies should not be quoted.
7. **`n_params` identifying the architecture at 1.000 is a fact about this pool** of five
   architectures with five distinct sizes. It is not a law.